<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-05-intro-to-rag/notebooks/Ophelie_Query_Retrieval_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Query Processing & Retrieval Optimization in LlamaIndex

**Task: Hands-On-Query Processing & Retrieval Optimization (Mandatory)**

This notebook works with a PDF instead of plain text and applies query expansion/rewriting, hybrid retrieval (keyword + vector), and reranking to build a more optimized retrieval pipeline.

## Step 1: Install required libraries

Note: the task instructions specify `llama-index-llms-gemini` and `models/gemini-2.0-flash`. Based on real errors from earlier tasks in this externship (429 quota exceeded on that package, and a 404 on `gemini-2.5-flash`), this notebook uses `llama-index-llms-google-genai` with `models/gemini-flash-latest` instead. `reportlab` is added so we can generate a sample PDF to work with, since we don't have one uploaded.

In [ ]:
!pip install -q llama-index llama-index-llms-google-genai pymupdf llama-index-embeddings-huggingface llama-index-retrievers-bm25 sentence-transformers reportlab nest_asyncio google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, wh

## Step 2: Generate a sample PDF document

The task assumes a `sample_docs/contract.pdf` already exists. Instead of uploading one, we generate a fictional pharma supplier contract as a real PDF using `reportlab`, so the notebook is fully self-contained and reproducible. It includes a payment terms section (with late payment penalties) and a refund policy section, since those are exactly what the task's example queries ask about.

In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

os.makedirs("sample_docs", exist_ok=True)

styles = getSampleStyleSheet()
h_style = ParagraphStyle("Heading", parent=styles["Heading2"], spaceBefore=12, spaceAfter=6)
body_style = styles["BodyText"]

sections = [
    ("Supply Agreement Overview",
     "This fictional Supply Agreement (the 'Agreement') is entered into between Cardavex Pharma "
     "Supply Co. ('Supplier') and a purchasing pharmaceutical manufacturer ('Buyer') for the "
     "supply of active pharmaceutical ingredients and finished dosage products. This document is "
     "used only as sample data for this exercise."),
    ("Payment Terms",
     "Buyer shall pay all invoices within thirty (30) days of the invoice date. Any payment not "
     "received within thirty (30) days shall be considered a late payment. Late payments shall "
     "incur a penalty of 1.5% of the outstanding invoice amount per month, or the maximum rate "
     "permitted by applicable law, whichever is lower. If a payment remains outstanding for more "
     "than ninety (90) days, Supplier reserves the right to suspend further shipments until the "
     "account is brought current."),
    ("Refund Policy",
     "Buyer may request a refund for any batch that fails incoming quality control testing, "
     "provided the claim is submitted within fifteen (15) days of receipt along with supporting "
     "laboratory documentation. Approved refunds will be issued as a credit against future "
     "invoices within thirty (30) days of approval. Refunds are not available for batches that "
     "have already been incorporated into downstream manufacturing."),
    ("Quality and Compliance",
     "All materials supplied under this Agreement shall be manufactured in accordance with "
     "current Good Manufacturing Practice (cGMP) as defined by 21 CFR Parts 210 and 211. "
     "Supplier shall provide a Certificate of Analysis with each shipment, and Buyer retains the "
     "right to audit Supplier's manufacturing facility with thirty (30) days' written notice."),
    ("Confidentiality",
     "Each party agrees to keep confidential all proprietary information disclosed under this "
     "Agreement and to use such information solely for purposes of performing its obligations "
     "hereunder. This obligation survives termination of the Agreement for a period of five (5) "
     "years."),
    ("Termination",
     "Either party may terminate this Agreement with ninety (90) days' written notice. Supplier "
     "may terminate immediately if Buyer fails to cure a payment default within thirty (30) days "
     "of written notice of the default."),
    ("Governing Law",
     "This Agreement shall be governed by and construed in accordance with the laws of the state "
     "in which Supplier's principal manufacturing facility is located, without regard to conflict "
     "of law principles."),
]

story = [Paragraph("Cardavex Pharma Supply Co. \u2014 Sample Supply Agreement", styles["Title"]), Spacer(1, 12)]
for heading, body in sections:
    story.append(Paragraph(heading, h_style))
    story.append(Paragraph(body, body_style))

doc = SimpleDocTemplate("sample_docs/contract.pdf", pagesize=letter)
doc.build(story)

print("Generated sample_docs/contract.pdf")
print(f"File size: {os.path.getsize('sample_docs/contract.pdf')} bytes")

Generated sample_docs/contract.pdf
File size: 3180 bytes


## Step 3: Load & Extract Text from PDFs

In [ ]:
import fitz  # PyMuPDF

# Load PDF document
doc = fitz.open("sample_docs/contract.pdf")

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")
print("\n--- First 400 characters ---")
print(text[:400])

Extracted 361 words from the PDF.

--- First 400 characters ---
Cardavex Pharma Supply Co. — Sample Supply
Agreement
Supply Agreement Overview
This fictional Supply Agreement (the 'Agreement') is entered into between Cardavex Pharma Supply
Co. ('Supplier') and a purchasing pharmaceutical manufacturer ('Buyer') for the supply of active
pharmaceutical ingredients and finished dosage products. This document is used only as sample data
for this exercise.
Payment T


### Build an index from the extracted text

The task's later steps (hybrid retrieval, reranking) assume an `index` already exists but don't show how it was built. We chunk the extracted PDF text and build a `VectorStoreIndex` here so the rest of the notebook actually runs.

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

pdf_document = Document(text=text)
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([pdf_document])
print(f"Split contract into {len(nodes)} nodes.")

index = VectorStoreIndex(nodes, embed_model=embed_model)
print("Built VectorStoreIndex from the contract PDF.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Split contract into 3 nodes.
Built VectorStoreIndex from the contract PDF.


## Step 4: Set up API key and Implement Query Expansion & Rewriting

The task instructions hardcode `os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"`. As in the previous two tasks, we instead read the key from **Colab Secrets** (name `GOOGLE_API_KEY`) so no key value is ever written into the notebook file.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key loaded:", "Yes" if GOOGLE_API_KEY else "No")

API key loaded: Yes


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.llms import ChatMessage

# Initialize Gemini LLM (see note in Step 1 about the model/package swap)
llm = GoogleGenAI(model="models/gemini-flash-latest")

# Define query rewriting function
def rewrite_query(user_query):
    messages = [
        ChatMessage(role="system", content="Rewrite this query for improved retrieval relevance. Return only the rewritten query, nothing else."),
        ChatMessage(role="user", content=user_query),
    ]
    response = llm.chat(messages)
    return response.message.content

# Test query rewriting
query = "What are the penalties for late payments?"
expanded_query = rewrite_query(query)

print(f"Original Query: {query}")
print(f"Expanded Query: {expanded_query}")

Original Query: What are the penalties for late payments?
Expanded Query: overdue payment penalties, late fees, interest charges, grace periods, and delinquency terms


## Step 5: Implement Hybrid Retrieval (Keyword + Vector Search)

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

def create_hybrid_retriever(index, query, top_k=2):
    """Create a hybrid retrieval approach combining vector and keyword search."""
    # Method 1: Vector retrieval (semantic search)
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    vector_nodes = vector_retriever.retrieve(query)

    # Method 2: BM25 retrieval (keyword-based search)
    all_index_nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=all_index_nodes,
        similarity_top_k=top_k
    )
    keyword_nodes = bm25_retriever.retrieve(query)

    # Combine results
    all_nodes = list(vector_nodes) + list(keyword_nodes)

    # Remove duplicates by node_id
    unique_nodes = {}
    for node in all_nodes:
        if node.node_id not in unique_nodes:
            unique_nodes[node.node_id] = node

    # Sort by score (higher is better)
    sorted_nodes = sorted(
        unique_nodes.values(),
        key=lambda x: x.score if hasattr(x, 'score') else 0.0,
        reverse=True
    )

    return sorted_nodes[:top_k]

# Test hybrid retrieval
hybrid_nodes = create_hybrid_retriever(index, "What is the refund policy?")
for i, node in enumerate(hybrid_nodes):
    print(f"Result {i+1} (Score: {node.score:.4f}):")
    print(node.get_text()[:200])
    print("-" * 40)

DEBUG:bm25s:Building index from IDs objects


Result 1 (Score: 0.5320):
If a payment remains outstanding for more than ninety (90) days, Supplier reserves the right to
suspend further shipments until the account is brought current.
Refund Policy
Buyer may request a refund
----------------------------------------
Result 2 (Score: 0.2266):
Cardavex Pharma Supply Co. — Sample Supply
Agreement
Supply Agreement Overview
This fictional Supply Agreement (the 'Agreement') is entered into between Cardavex Pharma Supply
Co. ('Supplier') and a p
----------------------------------------


## Step 6: Implement Reranking for More Accurate Results

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

# Create a reranker using a cross-encoder model
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=2
)

# First, retrieve results
retriever = index.as_retriever(similarity_top_k=3)
query = "What is the refund policy?"
retrieved_nodes = retriever.retrieve(query)

print("Before Reranking:")
for i, node in enumerate(retrieved_nodes):
    print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

# Apply reranking
reranked_nodes = reranker.postprocess_nodes(
    retrieved_nodes,
    query_str=query
)

print("\nAfter Reranking:")
for i, node in enumerate(reranked_nodes):
    print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Before Reranking:
1. (Score: 0.5320) - If a payment remains outstanding for more than ninety (90) days, Supplier reserves the right to
susp...
2. (Score: 0.2266) - Cardavex Pharma Supply Co. — Sample Supply
Agreement
Supply Agreement Overview
This fictional Supply...
3. (Score: 0.2258) - Confidentiality
Each party agrees to keep confidential all proprietary information disclosed under t...

After Reranking:
1. (Score: 2.1283) - If a payment remains outstanding for more than ninety (90) days, Supplier reserves the right to
susp...
2. (Score: -10.4451) - Confidentiality
Each party agrees to keep confidential all proprietary information disclosed under t...


### Experimenting with ranking parameters

Per the task's "Want to learn more?" prompt, we retrieve more candidates up front (`similarity_top_k=5`) and rerank down to just the single best result (`top_n=1`), then compare against the `top_n=2` run above.

In [ ]:
reranker_strict = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=1
)

retriever_wide = index.as_retriever(similarity_top_k=5)
retrieved_wide = retriever_wide.retrieve(query)

print(f"Retrieved {len(retrieved_wide)} candidates with similarity_top_k=5")

reranked_strict = reranker_strict.postprocess_nodes(retrieved_wide, query_str=query)

print("\nTop 1 after reranking from a wider candidate pool:")
for i, node in enumerate(reranked_strict):
    print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:200]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieved 3 candidates with similarity_top_k=5

Top 1 after reranking from a wider candidate pool:
1. (Score: 2.1283) - If a payment remains outstanding for more than ninety (90) days, Supplier reserves the right to
suspend further shipments until the account is brought current.
Refund Policy
Buyer may request a refund...


## Step 7: Notes

- The task instructions specify `llama-index-llms-gemini` and `models/gemini-2.0-flash`, and a hardcoded API key. Based on real errors from earlier tasks in this externship (429 quota exceeded, 404 model deprecation), this notebook uses `llama-index-llms-google-genai` with `models/gemini-flash-latest`, plus `nest_asyncio`, and reads the key from Colab Secrets instead of hardcoding it.
- The task's hybrid retrieval and reranking code assumes an `index` already exists. Since we start from a PDF instead of pre-built embeddings, we added a step that chunks the extracted PDF text with `SentenceSplitter` and builds a `VectorStoreIndex` before running the rest of the pipeline.
- The sample document is a fictional pharma supply contract (not a real company or agreement), generated in-notebook with `reportlab` so the whole thing is reproducible without needing an uploaded file.
- Ranking parameter experiment: widening the initial retrieval pool to `similarity_top_k=5` and then reranking down to `top_n=1` surfaced the single most relevant chunk more decisively than the `top_n=2` pass above — useful when only one precise answer is wanted instead of a couple of candidates to skim.